<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/03_sort_compute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Sorting, limiting & computed columns

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~25 min**

## Learning objectives
- sort with `ORDER BY` (asc/desc) and take top-N with `LIMIT`/`OFFSET`;
- compute new columns with arithmetic and `ROUND`;
- create categories on the fly with `CASE WHEN`.

Order results, take the top-N, compute new columns, and label rows with `CASE`.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [1]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.6/397.6 kB 19.8 MB/s eta 0:00:00


Connecting to 'sqlite:///../data/omics.db'

Connected to omics.db — you are ready.


## 1. ORDER BY

In [2]:
%%sql
SELECT sample_id, ph FROM samples
WHERE ph IS NOT NULL
ORDER BY ph DESC;

Running query in 'sqlite:///../data/omics.db'

sample_id,ph
S015,7.9
S021,7.74
S016,7.69
S019,7.68
S022,7.55
S030,7.46
S017,7.43
S023,7.41
S020,7.39
S014,7.37


You can sort by several keys: the second breaks ties in the first. Ascending (`ASC`) is the default.

In [3]:
%%sql
-- sort by environment (A->Z), then coldest-first within each environment
SELECT sample_id, environment, temperature_c
FROM samples
WHERE temperature_c IS NOT NULL
ORDER BY environment ASC, temperature_c ASC;

Running query in 'sqlite:///../data/omics.db'

sample_id,environment,temperature_c
S019,Freshwater,15.6
S013,Freshwater,16.4
S015,Freshwater,17.1
S021,Freshwater,17.7
S017,Freshwater,20.0
S018,Freshwater,20.2
S024,Freshwater,20.7
S014,Freshwater,21.2
S023,Freshwater,22.1
S020,Freshwater,22.5


In [5]:
%%sql
-- DID IT MYSEL NOT WORKING
SELECT sample_id , environment, temperature_c, PH
FROM samples
WHERE temperature_c , ph IS NOT NULL
ORDER BY PH ASC ;

Running query in 'sqlite:///../data/omics.db'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(sqlite3.OperationalError) near ",": syntax error
[SQL: SELECT sample_id , environment, temperature_c, PH
FROM samples
WHERE temperature_c , ph IS NOT NULL
ORDER BY PH ASC ;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)



## 2. Top-N: ORDER BY + LIMIT
The 10 largest single observations in the study:

In [8]:
%%sql
SELECT sample_id, taxon_id, count
FROM abundance
ORDER BY count DESC
LIMIT 10;

Running query in 'sqlite:///../data/omics.db'

sample_id,taxon_id,count
S004,T020,391
S023,T003,365
S006,T021,326
S004,T027,306
S011,T021,276
S034,T007,260
S011,T003,255
S007,T012,235
S005,T027,227
S005,T004,226


## 3. OFFSET — skip rows (e.g. the second page)

In [7]:
%%sql
SELECT sample_id, ph FROM samples
WHERE ph IS NOT NULL
ORDER BY ph DESC
LIMIT 5 OFFSET 5;

Running query in 'sqlite:///../data/omics.db'

sample_id,ph
S030,7.46
S017,7.43
S023,7.41
S020,7.39
S014,7.37


## 4. Computed columns
Convert temperature to Fahrenheit on the fly.

In [6]:
%%sql
SELECT sample_id, temperature_c,
       ROUND(temperature_c * 9.0 / 5 + 32, 1) AS temperature_f
FROM samples
LIMIT 8;

Running query in 'sqlite:///../data/omics.db'

sample_id,temperature_c,temperature_f
S001,14.5,58.1
S002,13.4,56.1
S003,13.5,56.3
S004,17.8,64.0
S005,19.1,66.4
S006,19.3,66.7
S007,17.6,63.7
S008,14.3,57.7


Computed columns can combine two columns. Here, a crude 'reads-per-cm-of-depth' style ratio (any arithmetic is allowed).

In [9]:
%%sql
-- combine two columns in one expression; NULLIF avoids divide-by-zero
SELECT sample_id, temperature_c, depth_cm,
       ROUND(temperature_c / NULLIF(depth_cm, 0), 2) AS temp_per_cm
FROM samples
WHERE depth_cm IS NOT NULL
LIMIT 8;

Running query in 'sqlite:///../data/omics.db'

sample_id,temperature_c,depth_cm,temp_per_cm
S001,14.5,9.3,1.56
S002,13.4,18.1,0.74
S003,13.5,19.3,0.7
S005,19.1,5.6,3.41
S006,19.3,13.3,1.45
S007,17.6,3.9,4.51
S008,14.3,4.2,3.4
S009,16.7,6.9,2.42


## 5. CASE — build a category
Label each sample by its pH.

In [10]:
%%sql
SELECT sample_id, ph,
       CASE WHEN ph < 6.5 THEN 'acidic'
            WHEN ph > 7.5 THEN 'alkaline'
            ELSE 'neutral' END AS reaction
FROM samples
WHERE ph IS NOT NULL;

Running query in 'sqlite:///../data/omics.db'

sample_id,ph,reaction
S001,5.88,acidic
S002,6.46,acidic
S003,6.09,acidic
S004,6.17,acidic
S005,6.04,acidic
S006,6.01,acidic
S007,6.46,acidic
S008,5.71,acidic
S010,5.85,acidic
S011,6.4,acidic


A different `CASE`: bucket a *numeric* column (temperature) into named classes.

In [ ]:
%%sql
-- classify each sample's thermal regime
SELECT sample_id, temperature_c,
       CASE WHEN temperature_c < 10 THEN 'cold'
            WHEN temperature_c < 20 THEN 'temperate'
            ELSE 'warm' END AS thermal_class
FROM samples
WHERE temperature_c IS NOT NULL
ORDER BY temperature_c;

---
## Exercises

**Exercise.** Show the 5 warmest samples (highest `temperature_c`).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, temperature_c FROM samples
ORDER BY temperature_c DESC
LIMIT 5;
```
</details>

**Exercise.** List sample_id and pH rounded to 1 decimal, for samples with a pH.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, ROUND(ph, 1) AS ph_1dp FROM samples
WHERE ph IS NOT NULL;
```
</details>

**Exercise.** Label each sample `shallow` (depth < 10) or `deep` with CASE.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, depth_cm,
       CASE WHEN depth_cm < 10 THEN 'shallow' ELSE 'deep' END AS layer
FROM samples
WHERE depth_cm IS NOT NULL;
```
</details>

### Recap
- `ORDER BY col DESC` sorts; `LIMIT`/`OFFSET` take top-N / pages.
- You can compute columns with arithmetic and `ROUND(x, n)`.
- `CASE WHEN … THEN … ELSE … END` builds categories.
- Next: 04 · Aggregation with GROUP BY.